# Regressão Linear

In [30]:
# -*- coding: utf-8 -*-
"""Regressão Linear para Previsão de Séries Temporais - Versão Corrigida

Este código implementa as recomendações dos artigos:
- Zeng et al. (2023): Uso de LTSF-Linear (apenas valores defasados da própria série)
- Toner & Darlow (2024): Solução OLS de forma fechada, sem validação cruzada embaralhada
- Ajiono & Hariguna (2023): Avaliação com MAD, MSE, MAPE e validação de resíduos

Principais correições:
1. Criação de defasagens (lags) da variável alvo (evita data leakage)
2. Divisão treino-teste estritamente sequencial (shuffle=False)
3. Remoção de GridSearchCV (substituído por OLS padrão)
4. Uso de TimeSeriesSplit para validação (opcional, mas não necessário para OLS)
5. Cálculo de MAD e MAPE, além de MSE e RMSE
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. Carregamento e preparação dos dados
# ============================================================================
data = pd.read_csv("/content/carne-brasileira-exportada.csv", encoding='utf-8', sep=',')
print("Dados originais:")
print(data.head(), "\n")

# ============================================================================
# 2. Função para criar defasagens (lags) da variável alvo
#    Abordagem LTSF-Linear (Zeng et al., 2023): usa apenas valores passados
# ============================================================================
def create_lagged_series(df, target_col, n_lags, group_col=None):
    """
    Cria um conjunto de features baseado nas defasagens da própria série.
    - target_col: nome da coluna alvo (ex: 'exportacao_usd')
    - n_lags: número de defasagens a incluir (ex: 4)
    - group_col: se houver múltiplas séries (ex: 'tipo_carne'), criar lags por grupo
    Retorna X (lags) e y (próximo valor), preservando ordem temporal.
    """
    df = df.copy()
    if group_col:
        # Aplica shift separadamente para cada grupo
        lagged_dfs = []
        for _, group in df.groupby(group_col):
            group = group.sort_values(['ano', 'trimestre'])
            for lag in range(1, n_lags + 1):
                group[f'lag_{lag}'] = group[target_col].shift(lag)
            lagged_dfs.append(group)
        df = pd.concat(lagged_dfs, ignore_index=True)
    else:
        for lag in range(1, n_lags + 1):
            df[f'lag_{lag}'] = df[target_col].shift(lag)
    
    # Remove linhas com NaN (primeiras n_lags observações de cada grupo)
    df = df.dropna().reset_index(drop=True)
    
    # Features = colunas de lag, target = valor atual
    feature_cols = [f'lag_{i}' for i in range(1, n_lags + 1)]
    X = df[feature_cols]
    y = df[target_col]
    return X, y, df[['ano', 'trimestre', group_col]] if group_col else df[['ano', 'trimestre']]

# ============================================================================
# 3. Avaliação por tipo de carne (bovino, suino, frango)
#    Usando apenas lags da própria série - método LTSF-Linear
# ============================================================================
n_lags = 4  # número de trimestres passados usados para prever o próximo

resultados = []

for carne in ['bovino', 'suino', 'frango']:
    print(f"\n{'='*50}")
    print(f"Processando: {carne.upper()}")
    print(f"{'='*50}")
    
    # Filtra a série específica
    df_carne = data[data['tipo_carne'] == carne].copy()
    df_carne = df_carne.sort_values(['ano', 'trimestre']).reset_index(drop=True)
    
    # Cria defasagens (features) da própria variável alvo 'exportacao_usd'
    X, y, info = create_lagged_series(df_carne, target_col='exportacao_usd', 
                                      n_lags=n_lags, group_col=None)
    
    # Divisão treino-teste sequencial (80% treino, 20% teste)
    n_total = len(X)
    n_train = int(n_total * 0.8)
    X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
    y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]
    
    print(f"Total de amostras (após criar lags): {n_total}")
    print(f"Treino: {len(X_train)}  | Teste: {len(X_test)}")
    
    # ========================================================================
    # 4. Treinamento do modelo OLS (sem restrições artificiais)
    #    Seguindo Toner & Darlow (2024): solução de forma fechada
    # ========================================================================
    model = LinearRegression(fit_intercept=True, positive=False, copy_X=True)
    model.fit(X_train, y_train)
    
    # Predições
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # ========================================================================
    # 5. Métricas de erro (conforme Ajiono & Hariguna, 2023)
    # ========================================================================
    # Treino
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    # Evita divisão por zero para MAPE
    mape_train = np.mean(np.abs((y_train - y_pred_train) / (y_train + 1e-8))) * 100
    
    # Teste
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mape_test = np.mean(np.abs((y_test - y_pred_test) / (y_test + 1e-8))) * 100
    
    # R²
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    
    # Coeficientes
    coefs = model.coef_
    
    print(f"\n--- Resultados para {carne} ---")
    print(f"Intercept: {model.intercept_:.4f}")
    print(f"Coeficientes (lag1..lag{n_lags}): {coefs}")
    print("\nMétricas de Treino:")
    print(f"  MSE : {mse_train:.6f}  | RMSE: {rmse_train:.6f}  | MAE: {mae_train:.6f}  | MAPE: {mape_train:.2f}%  | R²: {r2_train:.4f}")
    print("Métricas de Teste:")
    print(f"  MSE : {mse_test:.6f}  | RMSE: {rmse_test:.6f}  | MAE: {mae_test:.6f}  | MAPE: {mape_test:.2f}%  | R²: {r2_test:.4f}")
    
    # Armazena para comparação final
    resultados.append({
        'carne': carne,
        'n_lags': n_lags,
        'MSE_teste': mse_test,
        'RMSE_teste': rmse_test,
        'MAE_teste': mae_test,
        'MAPE_teste': mape_test,
        'R²_teste': r2_test
    })
    
    # Opcional: mostrar últimos valores previstos vs reais
    print("\nÚltimos 5 valores reais vs previstos (teste):")
    compare = pd.DataFrame({'Real': y_test.values[-5:], 
                            'Previsto': y_pred_test[-5:]})
    print(compare)

# ============================================================================
# 6. Sumário comparativo dos três tipos de carne
# ============================================================================
print("\n\n" + "="*70)
print("SUMÁRIO COMPARATIVO (Teste)")
print("="*70)
df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))

# ============================================================================
# 7. (Opcional) Validação com TimeSeriesSplit - apenas para demonstrar
#    Nota: Toner & Darlow mostram que OLS tem solução única, então 
#    validação cruzada não é necessária, mas pode ser usada para estimar variância.
# ============================================================================
from sklearn.model_selection import TimeSeriesSplit

print("\n\n" + "="*70)
print("VALIDACÃO CRUZADA TEMPORAL (TimeSeriesSplit) - Exemplo para bovino")
print("="*70)
# Obtém dados bovino novamente
df_bov = data[data['tipo_carne'] == 'bovino'].copy()
X_bov, y_bov, _ = create_lagged_series(df_bov, 'exportacao_usd', n_lags=4, group_col=None)

tscv = TimeSeriesSplit(n_splits=3)
mse_scores = []

for i, (train_idx, test_idx) in enumerate(tscv.split(X_bov)):
    X_tr, X_te = X_bov.iloc[train_idx], X_bov.iloc[test_idx]
    y_tr, y_te = y_bov.iloc[train_idx], y_bov.iloc[test_idx]
    
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mse = mean_squared_error(y_te, y_pred)
    mse_scores.append(mse)
    print(f"Fold {i+1}: MSE = {mse:.6f}")

print(f"MSE médio da TimeSeriesCV: {np.mean(mse_scores):.6f} (+/- {np.std(mse_scores):.6f})")
print("(Esta validação é apenas informativa; a solução OLS global é única e convexa.)")

Dados originais:
    ano  trimestre tipo_carne  producao_cabecas_anual_mi  abate_cabecas  \
0  2014          1     bovino                     212.34          8.366   
1  2014          2     bovino                     212.34          8.516   
2  2014          3     bovino                     212.34          8.456   
3  2014          4     bovino                     212.34          8.525   
4  2014          1      suino                      37.93          8.686   

   peso_carcaca_ton  exportacao_usd  media_cambio_usdbrl  custo_milho_rs  \
0             1.951           1.345                 2.34           22.57   
1             2.006           1.384                 2.22           23.59   
2             2.036           1.547                 2.28           19.97   
3             2.058           1.519                 2.59           21.46   
4             0.747           0.260                 2.34           22.57   

   custo_soja_rs  preco_medio_ton_usd  
0          62.01                4.4